# Module 3: Customer Intent Classifier

This notebook trains a supervised multi-class intent routing classifier on the `bitext/Bitext-customer-support-llm-chatbot-training-dataset`, condensing 27 fine-grained retail intents into 7 core operational routing categories.

In [ ]:
import re
import unicodedata
import pickle
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## 1. Load Bitext Customer Support Dataset

In [ ]:
raw_dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df = pd.DataFrame(raw_dataset)
print('Total instructions:', len(df))
print('Original unique intents:', len(df['intent'].unique()))
print(df[['instruction', 'category', 'intent']].head())

## 2. Intent Categorization into 7 Routing Buckets

In [ ]:
def map_intent_category(fine_intent):
    intent_mapping = {
        'track_order': 'order_status',
        'delivery_options': 'order_status',
        'delivery_period': 'order_status',
        
        'cancel_order': 'order_management',
        'change_order': 'order_management',
        'place_order': 'order_management',
        'change_shipping_address': 'order_management',
        'set_up_shipping_address': 'order_management',
        'check_cancellation_fee': 'order_management',
        
        'check_invoice': 'billing_and_refunds',
        'get_invoice': 'billing_and_refunds',
        'get_refund': 'billing_and_refunds',
        'track_refund': 'billing_and_refunds',
        'check_refund_policy': 'billing_and_refunds',
        'payment_issue': 'billing_and_refunds',
        'check_payment_methods': 'billing_and_refunds',
        
        'create_account': 'account_management',
        'edit_account': 'account_management',
        'delete_account': 'account_management',
        'switch_account': 'account_management',
        'recover_password': 'account_management',
        'registration_problems': 'account_management',
        'newsletter_subscription': 'account_management',
        
        'complaint': 'complaint',
        'review': 'complaint',
        'contact_customer_service': 'complaint',
        'contact_human_agent': 'complaint'
    }
    return intent_mapping.get(fine_intent, 'out_of_scope')

df['mapped_intent'] = df['intent'].apply(map_intent_category)
print(df['mapped_intent'].value_counts())

## 3. Dataset Augmentation for Greetings and Out-of-Scope

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize('NFKC', str(text))
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

df['clean_text'] = df['instruction'].apply(normalize_text)

greetings = [
    'hello', 'hi there', 'hey', 'good morning', 'good afternoon',
    'good evening', 'hi bot', 'hey support', 'thank you so much',
    'thanks for the help', 'appreciate it', 'goodbye', 'bye', 'see you later'
]
out_of_scope = [
    'what is the capital of italy', 'tell me a funny joke',
    'who won the world cup in 1998', 'how to write python code',
    'what is the weather like today', 'can you write an essay about space',
    'how tall is mount everest', 'who is the president of france'
]

extra_rows = []
for g in greetings:
    extra_rows.append({'clean_text': normalize_text(g), 'mapped_intent': 'greeting'})
for o in out_of_scope:
    extra_rows.append({'clean_text': normalize_text(o), 'mapped_intent': 'out_of_scope'})
    
df_combined = pd.concat([df[['clean_text', 'mapped_intent']], pd.DataFrame(extra_rows)], ignore_index=True)

X_train, X_test, y_train, y_test = train_test_split(
    df_combined['clean_text'],
    df_combined['mapped_intent'],
    test_size=0.15,
    random_state=42,
    stratify=df_combined['mapped_intent']
)

## 4. Pipeline Training and Evaluation

In [ ]:
intent_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=25000,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        C=3.0,
        max_iter=1000,
        solver='lbfgs'
    ))
])

intent_pipeline.fit(X_train, y_train)
preds = intent_pipeline.predict(X_test)

print(f'Test Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%\n')
print(classification_report(y_test, preds))

## 5. Live Routing Validation

In [ ]:
test_queries = [
    'Hello good morning', 
    'Where is my shipment right now?',
    'I need to cancel my order immediately.',
    'Can I receive a full refund on this item?',
    'I forgot my account password and need to reset it.',
    'I want to speak with a human agent, your service is terrible!',
    'What is the speed of light?'
]

for q in test_queries:
    clean = normalize_text(q)
    predicted_intent = intent_pipeline.predict([clean])[0]
    print(f'Query: "{q}" -> Intent: {predicted_intent}')